# Notebook 06 — Cross-Category Factor Structure

**Assignment**: Global structure family  
**Lenses**: Full 50×50 correlation heatmap, hierarchical clustering, PCA scree + loadings  
**Days**: 2, 3, 4  

**Goal**: Identify products that cluster outside their named category (potential mislabeling or hidden factor structure), and determine whether PC1/PC2/PC3 loadings map cleanly to categories.

All analysis is read-only on `data/`. No strategy proposals, no PnL predictions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/Users/bensinek/Documents/Coding/Prosperity4/data/round_5/prices'
DAYS = [2, 3, 4]

# Load and concatenate all days
frames = []
for d in DAYS:
    df = pd.read_csv(f'{DATA_DIR}/prices_round_5_day_{d}.csv', sep=';')
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)

print(f'Total rows: {len(raw):,}')
print(f'Products: {raw["product"].nunique()}')
print(f'Days: {sorted(raw["day"].unique())}')
print(f'Timestamps per (day, product):')
print(raw.groupby(['day','product'])['timestamp'].count().describe())


Total rows: 1,500,000
Products: 50
Days: [np.int64(2), np.int64(3), np.int64(4)]
Timestamps per (day, product):
count      150.0
mean     10000.0
std          0.0
min      10000.0
25%      10000.0
50%      10000.0
75%      10000.0
max      10000.0
Name: timestamp, dtype: float64


## 1. Build Mid-Price Return Matrix

For correlation and PCA we use **tick-level log-returns of mid_price**, pooled across all three days. We assign each product a global tick index = day * 10000 + timestamp so the series are aligned.

For the 50×50 heatmap we also compute correlations on price **levels** (z-scored within each day) as a complementary view, since many products may be mean-reverting (returns near zero) but co-integrated in levels.


In [2]:
# Assign a global tick index so all days are concatenated in order
raw['global_tick'] = raw['day'] * 10000 + raw['timestamp']

# Pivot to wide form: rows = global_tick, cols = product
mid_wide = raw.pivot_table(index='global_tick', columns='product', values='mid_price')
mid_wide = mid_wide.sort_index()

print(f'Wide matrix shape: {mid_wide.shape}')
print(f'Missing values per column (should be 0):')
print(mid_wide.isnull().sum().describe())


Wide matrix shape: (10200, 50)
Missing values per column (should be 0):
count    50.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
dtype: float64


In [3]:
# --- Category mapping ---
CATEGORIES = {
    'GALAXY_SOUNDS': ['GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_BLACK_HOLES',
                      'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_WINDS',
                      'GALAXY_SOUNDS_SOLAR_FLAMES'],
    'SLEEP_POD':     ['SLEEP_POD_SUEDE', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_POLYESTER',
                      'SLEEP_POD_NYLON', 'SLEEP_POD_COTTON'],
    'MICROCHIP':     ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE',
                      'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE'],
    'PEBBLES':       ['PEBBLES_XS', 'PEBBLES_S', 'PEBBLES_M', 'PEBBLES_L', 'PEBBLES_XL'],
    'ROBOT':         ['ROBOT_VACUUMING', 'ROBOT_MOPPING', 'ROBOT_DISHES',
                      'ROBOT_LAUNDRY', 'ROBOT_IRONING'],
    'UV_VISOR':      ['UV_VISOR_YELLOW', 'UV_VISOR_AMBER', 'UV_VISOR_ORANGE',
                      'UV_VISOR_RED', 'UV_VISOR_MAGENTA'],
    'TRANSLATOR':    ['TRANSLATOR_SPACE_GRAY', 'TRANSLATOR_ASTRO_BLACK',
                      'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST',
                      'TRANSLATOR_VOID_BLUE'],
    'PANEL':         ['PANEL_1X2', 'PANEL_2X2', 'PANEL_1X4', 'PANEL_2X4', 'PANEL_4X4'],
    'OXYGEN_SHAKE':  ['OXYGEN_SHAKE_MORNING_BREATH', 'OXYGEN_SHAKE_EVENING_BREATH',
                      'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_CHOCOLATE', 'OXYGEN_SHAKE_GARLIC'],
    'SNACKPACK':     ['SNACKPACK_CHOCOLATE', 'SNACKPACK_VANILLA', 'SNACKPACK_PISTACHIO',
                      'SNACKPACK_STRAWBERRY', 'SNACKPACK_RASPBERRY'],
}

# Colors for categories
CAT_COLORS = {
    'GALAXY_SOUNDS': '#e6194b',
    'SLEEP_POD':     '#3cb44b',
    'MICROCHIP':     '#4363d8',
    'PEBBLES':       '#f58231',
    'ROBOT':         '#911eb4',
    'UV_VISOR':      '#42d4f4',
    'TRANSLATOR':    '#f032e6',
    'PANEL':         '#bfef45',
    'OXYGEN_SHAKE':  '#fabed4',
    'SNACKPACK':     '#469990',
}

# Build ordered product list (category-sorted) for heatmap
ORDERED_PRODUCTS = []
PRODUCT_CATEGORY = {}
for cat, prods in CATEGORIES.items():
    for p in prods:
        ORDERED_PRODUCTS.append(p)
        PRODUCT_CATEGORY[p] = cat

print('Category map built. Total products:', len(ORDERED_PRODUCTS))


Category map built. Total products: 50


In [4]:
# Compute log-returns (drop first tick of each day boundary to avoid cross-day artifacts)
# We compute returns per day then concatenate
returns_frames = []
levels_frames = []
for d in DAYS:
    sub = raw[raw['day'] == d].pivot_table(index='timestamp', columns='product', values='mid_price').sort_index()
    rets = np.log(sub).diff().iloc[1:]  # drop first row (NaN)
    # z-score levels within day
    lvls = (sub - sub.mean()) / sub.std()
    returns_frames.append(rets)
    levels_frames.append(lvls)

returns_all = pd.concat(returns_frames, ignore_index=True)
levels_all  = pd.concat(levels_frames, ignore_index=True)

print(f'Returns matrix: {returns_all.shape}  (rows = ticks, cols = products)')
print(f'Levels matrix:  {levels_all.shape}')
print(f'NaN in returns: {returns_all.isnull().sum().sum()}')


Returns matrix: (29997, 50)  (rows = ticks, cols = products)
Levels matrix:  (30000, 50)
NaN in returns: 0


## 2. Full 50×50 Correlation Heatmap (Returns)

Products ordered by named category. Red blocks along the diagonal = within-category correlation. Off-diagonal blocks = cross-category correlation. Unexpected off-diagonal hot spots signal hidden factor structure.


In [5]:
def plot_corr_heatmap(corr_df, title, ordered_products, product_category, cat_colors, save_path, figsize=(18, 16)):
    """Plot annotated 50x50 correlation heatmap with category color bars."""
    C = corr_df.loc[ordered_products, ordered_products].values
    n = len(ordered_products)
    
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(C, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    
    # Axis labels — short names
    short = [p.replace('GALAXY_SOUNDS_', 'GS_').replace('SLEEP_POD_', 'SP_')
              .replace('MICROCHIP_', 'MC_').replace('PEBBLES_', 'PB_')
              .replace('ROBOT_', 'RB_').replace('UV_VISOR_', 'UV_')
              .replace('TRANSLATOR_', 'TR_').replace('PANEL_', 'PN_')
              .replace('OXYGEN_SHAKE_', 'OX_').replace('SNACKPACK_', 'SN_')
              for p in ordered_products]
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(short, rotation=90, fontsize=6)
    ax.set_yticklabels(short, fontsize=6)
    
    # Color-coded category blocks — draw rectangles on the diagonal
    cat_boundaries = {}
    cur_cat = None
    start = 0
    for i, p in enumerate(ordered_products):
        cat = product_category[p]
        if cat != cur_cat:
            if cur_cat is not None:
                cat_boundaries[cur_cat] = (start, i - 1)
            cur_cat = cat
            start = i
    cat_boundaries[cur_cat] = (start, n - 1)
    
    for cat, (s, e) in cat_boundaries.items():
        rect = plt.Rectangle((s - 0.5, s - 0.5), e - s + 1, e - s + 1,
                               linewidth=2, edgecolor=cat_colors[cat],
                               facecolor='none', zorder=3)
        ax.add_patch(rect)
    
    # Legend
    handles = [mpatches.Patch(color=cat_colors[c], label=c) for c in cat_colors]
    ax.legend(handles=handles, bbox_to_anchor=(1.18, 1), loc='upper left', fontsize=7)
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

import os
os.makedirs('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots', exist_ok=True)

corr_returns = returns_all[ORDERED_PRODUCTS].corr()
print('Returns correlation matrix shape:', corr_returns.shape)
print('Min off-diagonal corr:', corr_returns.values[corr_returns.values < 1.0 - 1e-9].min().round(4))
print('Max off-diagonal corr:', corr_returns.values[corr_returns.values < 1.0 - 1e-9].max().round(4))

plot_corr_heatmap(corr_returns, 'Full 50×50 Return Correlation Heatmap (Days 2+3+4)',
                  ORDERED_PRODUCTS, PRODUCT_CATEGORY, CAT_COLORS,
                  save_path='/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/06_heatmap_returns.png')


Returns correlation matrix shape: (50, 50)
Min off-diagonal corr: -0.9232
Max off-diagonal corr: 0.9126


## 3. Full 50×50 Correlation Heatmap (Levels, z-scored within day)

Returns correlations may be near zero for hardcoded-FV products (tiny returns). Level correlations capture structural co-movement at longer horizons.


In [6]:
corr_levels = levels_all[ORDERED_PRODUCTS].corr()
print('Levels correlation matrix shape:', corr_levels.shape)
print('Min off-diagonal corr:', corr_levels.values[corr_levels.values < 1.0 - 1e-9].min().round(4))
print('Max off-diagonal corr:', corr_levels.values[corr_levels.values < 1.0 - 1e-9].max().round(4))

plot_corr_heatmap(corr_levels, 'Full 50×50 Level Correlation Heatmap (z-scored, Days 2+3+4)',
                  ORDERED_PRODUCTS, PRODUCT_CATEGORY, CAT_COLORS,
                  save_path='/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/06_heatmap_levels.png')


Levels correlation matrix shape: (50, 50)
Min off-diagonal corr: -0.9718
Max off-diagonal corr: 0.7404


## 4. Within-Category vs Cross-Category Correlation Summary

Quantify how tightly each category clusters vs. how much it leaks into other categories.


In [7]:
def within_cross_summary(corr_df, categories, label='returns'):
    rows = []
    all_products = [p for prods in categories.values() for p in prods]
    for cat, prods in categories.items():
        # within
        within_vals = []
        for i, pi in enumerate(prods):
            for j, pj in enumerate(prods):
                if i < j:
                    within_vals.append(corr_df.loc[pi, pj])
        # cross
        other = [p for p in all_products if p not in prods]
        cross_vals = [corr_df.loc[pi, pj] for pi in prods for pj in other]
        rows.append({
            'category': cat,
            f'within_mean_{label}': np.mean(within_vals),
            f'within_min_{label}': np.min(within_vals),
            f'within_max_{label}': np.max(within_vals),
            f'cross_mean_{label}': np.mean(cross_vals),
            f'cross_max_abs_{label}': np.max(np.abs(cross_vals)),
        })
    return pd.DataFrame(rows).set_index('category')

summary_ret = within_cross_summary(corr_returns, CATEGORIES, 'ret')
summary_lvl = within_cross_summary(corr_levels, CATEGORIES, 'lvl')
summary = pd.concat([summary_ret, summary_lvl], axis=1)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
print(summary.to_string())


               within_mean_ret  within_min_ret  within_max_ret  cross_mean_ret  cross_max_abs_ret  within_mean_lvl  within_min_lvl  within_max_lvl  cross_mean_lvl  cross_max_abs_lvl
category                                                                                                                                                                             
GALAXY_SOUNDS           0.0040         -0.0013          0.0130          0.0051             0.0254          -0.0532         -0.3231          0.2365          0.0019             0.7220
SLEEP_POD               0.0028         -0.0075          0.0090          0.0034             0.0206           0.1346         -0.3284          0.7404         -0.0030             0.7190
MICROCHIP               0.0062          0.0018          0.0126         -0.0006             0.0156          -0.0187         -0.5226          0.4565         -0.0010             0.6560
PEBBLES                -0.1913         -0.5059          0.0156         -0.0000            

## 5. Hierarchical Clustering Dendrogram (Returns)

Ward linkage on the distance matrix `D = 1 - |corr|`. Products that cluster together share return dynamics. Categories that stay tight = internally coherent. Products that migrate away = outliers or cross-category factors.


In [8]:
def plot_dendrogram(corr_df, ordered_products, product_category, cat_colors, title, save_path):
    # Distance matrix: 1 - |correlation|
    C = corr_df.loc[ordered_products, ordered_products].values
    D = 1 - np.abs(C)
    np.fill_diagonal(D, 0)
    # Ensure symmetry
    D = (D + D.T) / 2
    
    condensed = squareform(D)
    Z = linkage(condensed, method='ward')
    
    fig, ax = plt.subplots(figsize=(22, 8))
    
    short = [p.replace('GALAXY_SOUNDS_', 'GS_').replace('SLEEP_POD_', 'SP_')
              .replace('MICROCHIP_', 'MC_').replace('PEBBLES_', 'PB_')
              .replace('ROBOT_', 'RB_').replace('UV_VISOR_', 'UV_')
              .replace('TRANSLATOR_', 'TR_').replace('PANEL_', 'PN_')
              .replace('OXYGEN_SHAKE_', 'OX_').replace('SNACKPACK_', 'SN_')
              for p in ordered_products]
    
    ddata = dendrogram(Z, labels=short, ax=ax, leaf_rotation=90, leaf_font_size=7,
                       color_threshold=0)
    
    # Color the x tick labels by category
    for tick_label in ax.get_xticklabels():
        txt = tick_label.get_text()
        # reverse map short to product
        full = None
        for p, s in zip(ordered_products, short):
            if s == txt:
                full = p
                break
        if full:
            cat = product_category[full]
            tick_label.set_color(cat_colors[cat])
    
    handles = [mpatches.Patch(color=cat_colors[c], label=c) for c in cat_colors]
    ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
    ax.set_title(title, fontsize=13)
    ax.set_ylabel('Ward distance (1 - |corr|)')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    return Z, ddata

Z_ret, ddata_ret = plot_dendrogram(
    corr_returns, ORDERED_PRODUCTS, PRODUCT_CATEGORY, CAT_COLORS,
    'Hierarchical Clustering — Returns (Ward, D = 1 - |corr|)',
    '/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/06_dendrogram_returns.png'
)


## 6. Hierarchical Clustering — Level-based

Repeat dendrogram using price-level correlations. Some categories may only separate in levels (e.g. Pebbles size-gradient products).


In [9]:
Z_lvl, ddata_lvl = plot_dendrogram(
    corr_levels, ORDERED_PRODUCTS, PRODUCT_CATEGORY, CAT_COLORS,
    'Hierarchical Clustering — Price Levels (Ward, D = 1 - |corr|)',
    '/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/06_dendrogram_levels.png'
)


## 7. Cluster Membership — Do Products Stray From Their Category?

Cut the Ward dendrogram at 10 clusters and compare cluster membership to named categories. Products in a cluster dominated by a different category = potential mislabeling or cross-category factor.


In [10]:
def cluster_membership_report(Z, ordered_products, product_category, n_clusters=10, label=''):
    labels = fcluster(Z, n_clusters, criterion='maxclust')
    membership = pd.DataFrame({
        'product': ordered_products,
        'named_category': [product_category[p] for p in ordered_products],
        f'cluster_{label}': labels
    })
    
    # For each cluster, show what named categories are present
    print(f'\n--- Cluster composition ({label}, k={n_clusters}) ---')
    for cl in sorted(membership[f'cluster_{label}'].unique()):
        sub = membership[membership[f'cluster_{label}'] == cl]
        cats = sub['named_category'].value_counts()
        majority = cats.index[0]
        strays = sub[sub['named_category'] != majority]
        print(f'\nCluster {cl} (n={len(sub)}): majority={majority}')
        for _, row in sub.iterrows():
            flag = ' *** STRAY ***' if row['named_category'] != majority else ''
            print(f'  {row["product"]}{flag}')
    
    # Summary: products that ended up in a cluster dominated by another category
    stray_rows = []
    for cl in sorted(membership[f'cluster_{label}'].unique()):
        sub = membership[membership[f'cluster_{label}'] == cl]
        cats = sub['named_category'].value_counts()
        majority = cats.index[0]
        for _, row in sub.iterrows():
            if row['named_category'] != majority:
                stray_rows.append({
                    'product': row['product'],
                    'named_category': row['named_category'],
                    'cluster_majority': majority,
                    'cluster_id': cl
                })
    if stray_rows:
        print(f'\nSTRAY PRODUCTS ({label}):')
        print(pd.DataFrame(stray_rows).to_string(index=False))
    else:
        print(f'\nNo stray products detected ({label}) — all clusters map cleanly to named categories.')
    return membership

mem_ret = cluster_membership_report(Z_ret, ORDERED_PRODUCTS, PRODUCT_CATEGORY, n_clusters=10, label='ret')



--- Cluster composition (ret, k=10) ---

Cluster 1 (n=3): majority=SNACKPACK
  SNACKPACK_PISTACHIO
  SNACKPACK_STRAWBERRY
  SNACKPACK_RASPBERRY

Cluster 2 (n=5): majority=PEBBLES
  PEBBLES_XS
  PEBBLES_S
  PEBBLES_M
  PEBBLES_L
  PEBBLES_XL

Cluster 3 (n=2): majority=SNACKPACK
  SNACKPACK_CHOCOLATE
  SNACKPACK_VANILLA

Cluster 4 (n=7): majority=MICROCHIP
  SLEEP_POD_NYLON *** STRAY ***
  MICROCHIP_SQUARE
  MICROCHIP_RECTANGLE
  ROBOT_MOPPING *** STRAY ***
  TRANSLATOR_SPACE_GRAY *** STRAY ***
  PANEL_4X4 *** STRAY ***
  OXYGEN_SHAKE_MINT *** STRAY ***

Cluster 5 (n=7): majority=MICROCHIP
  MICROCHIP_CIRCLE
  MICROCHIP_OVAL
  ROBOT_VACUUMING *** STRAY ***
  ROBOT_IRONING *** STRAY ***
  UV_VISOR_AMBER *** STRAY ***
  PANEL_1X4 *** STRAY ***
  OXYGEN_SHAKE_CHOCOLATE *** STRAY ***

Cluster 6 (n=5): majority=GALAXY_SOUNDS
  GALAXY_SOUNDS_BLACK_HOLES
  GALAXY_SOUNDS_SOLAR_WINDS
  TRANSLATOR_GRAPHITE_MIST *** STRAY ***
  TRANSLATOR_VOID_BLUE *** STRAY ***
  OXYGEN_SHAKE_EVENING_BREATH *** S

In [11]:
mem_lvl = cluster_membership_report(Z_lvl, ORDERED_PRODUCTS, PRODUCT_CATEGORY, n_clusters=10, label='lvl')



--- Cluster composition (lvl, k=10) ---

Cluster 1 (n=2): majority=SNACKPACK
  SNACKPACK_STRAWBERRY
  SNACKPACK_RASPBERRY

Cluster 2 (n=5): majority=MICROCHIP
  MICROCHIP_OVAL
  PEBBLES_M *** STRAY ***
  ROBOT_IRONING *** STRAY ***
  TRANSLATOR_ECLIPSE_CHARCOAL *** STRAY ***
  SNACKPACK_PISTACHIO *** STRAY ***

Cluster 3 (n=6): majority=SLEEP_POD
  GALAXY_SOUNDS_SOLAR_WINDS *** STRAY ***
  SLEEP_POD_LAMB_WOOL
  SLEEP_POD_NYLON
  ROBOT_VACUUMING *** STRAY ***
  PANEL_1X2 *** STRAY ***
  OXYGEN_SHAKE_MORNING_BREATH *** STRAY ***

Cluster 4 (n=8): majority=MICROCHIP
  GALAXY_SOUNDS_DARK_MATTER *** STRAY ***
  SLEEP_POD_POLYESTER *** STRAY ***
  SLEEP_POD_COTTON *** STRAY ***
  MICROCHIP_CIRCLE
  MICROCHIP_SQUARE
  MICROCHIP_RECTANGLE
  ROBOT_MOPPING *** STRAY ***
  UV_VISOR_YELLOW *** STRAY ***

Cluster 5 (n=4): majority=GALAXY_SOUNDS
  GALAXY_SOUNDS_SOLAR_FLAMES
  ROBOT_LAUNDRY *** STRAY ***
  TRANSLATOR_SPACE_GRAY *** STRAY ***
  PANEL_4X4 *** STRAY ***

Cluster 6 (n=4): majority=UV_VI

## 8. PCA — Scree Plot + PC1/PC2/PC3 Loadings

PCA on the return matrix (50 products). Scree = how many global factors exist. Loadings = which categories drive each factor. If a single PC explains most variance → one dominant global factor (market beta). If PCs map cleanly to categories → category = factor. Mixtures → hidden cross-category factors.


In [12]:
# PCA on returns (fill any remaining NaN with 0 — there should be none after construction)
X = returns_all[ORDERED_PRODUCTS].fillna(0).values

pca = PCA(n_components=min(50, X.shape[1]))
pca.fit(X)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print('Explained variance by PC:')
for i, (ev, cv) in enumerate(zip(explained[:15], cumulative[:15])):
    print(f'  PC{i+1:02d}: {ev:.4f}  cumulative: {cv:.4f}')

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(range(1, 21), explained[:20], color='steelblue', alpha=0.8)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('Scree Plot — Returns PCA (top 20 PCs)')
ax.axhline(1/50, color='red', linestyle='--', label='Random baseline (1/50)')
ax.legend()

ax = axes[1]
ax.plot(range(1, 21), cumulative[:20], marker='o', color='steelblue')
ax.axhline(0.9, color='orange', linestyle='--', label='90%')
ax.axhline(0.5, color='red', linestyle='--', label='50%')
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('Cumulative Variance — Returns PCA')
ax.legend()

plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/06_pca_scree.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print(f'\nPCs needed for 50% variance: {np.searchsorted(cumulative, 0.50) + 1}')
print(f'PCs needed for 90% variance: {np.searchsorted(cumulative, 0.90) + 1}')
print(f'PC1 alone explains: {explained[0]:.4f} ({explained[0]*100:.2f}%)')


Explained variance by PC:
  PC01: 0.1270  cumulative: 0.1270
  PC02: 0.0580  cumulative: 0.1850
  PC03: 0.0428  cumulative: 0.2278
  PC04: 0.0384  cumulative: 0.2662
  PC05: 0.0337  cumulative: 0.2999
  PC06: 0.0331  cumulative: 0.3330
  PC07: 0.0330  cumulative: 0.3660
  PC08: 0.0327  cumulative: 0.3988
  PC09: 0.0320  cumulative: 0.4308
  PC10: 0.0211  cumulative: 0.4519
  PC11: 0.0205  cumulative: 0.4723
  PC12: 0.0204  cumulative: 0.4927
  PC13: 0.0185  cumulative: 0.5111
  PC14: 0.0168  cumulative: 0.5279
  PC15: 0.0156  cumulative: 0.5435



PCs needed for 50% variance: 13
PCs needed for 90% variance: 40
PC1 alone explains: 0.1270 (12.70%)


## 9. PC1 / PC2 / PC3 Loadings — Which Products Drive Each Factor?

High-magnitude loading = product is strongly aligned with that PC. Color = named category. If loadings are monochromatic (one category dominates a PC) → category = factor. Mixed loadings → cross-category factor or noise.


In [13]:
def plot_loadings(pca, ordered_products, product_category, cat_colors, n_pcs=5, save_path=None):
    """Plot loadings for top n_pcs as horizontal bar charts, colored by category."""
    short = [p.replace('GALAXY_SOUNDS_', 'GS_').replace('SLEEP_POD_', 'SP_')
              .replace('MICROCHIP_', 'MC_').replace('PEBBLES_', 'PB_')
              .replace('ROBOT_', 'RB_').replace('UV_VISOR_', 'UV_')
              .replace('TRANSLATOR_', 'TR_').replace('PANEL_', 'PN_')
              .replace('OXYGEN_SHAKE_', 'OX_').replace('SNACKPACK_', 'SN_')
              for p in ordered_products]
    colors = [cat_colors[product_category[p]] for p in ordered_products]
    
    fig, axes = plt.subplots(1, n_pcs, figsize=(5 * n_pcs, 14))
    if n_pcs == 1:
        axes = [axes]
    
    for pc_idx, ax in enumerate(axes):
        loadings = pca.components_[pc_idx]
        ev = pca.explained_variance_ratio_[pc_idx]
        # Sort by loading value
        order = np.argsort(loadings)
        sorted_loads = loadings[order]
        sorted_short = [short[i] for i in order]
        sorted_colors = [colors[i] for i in order]
        
        bars = ax.barh(range(len(sorted_loads)), sorted_loads, color=sorted_colors, alpha=0.85)
        ax.set_yticks(range(len(sorted_loads)))
        ax.set_yticklabels(sorted_short, fontsize=6.5)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title(f'PC{pc_idx+1} ({ev*100:.2f}% var)', fontsize=11)
        ax.set_xlabel('Loading')
    
    handles = [mpatches.Patch(color=cat_colors[c], label=c) for c in cat_colors]
    fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=8, bbox_to_anchor=(0.5, -0.04))
    plt.suptitle('PCA Loadings — Returns (top PCs)', fontsize=14, y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()

plot_loadings(pca, ORDERED_PRODUCTS, PRODUCT_CATEGORY, CAT_COLORS, n_pcs=5,
              save_path='/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/06_pca_loadings.png')


## 10. Dominant Category Per PC (Quantified)

For each PC, compute the average absolute loading per named category. The dominant category is the one with the highest mean |loading|.


In [14]:
loading_df = pd.DataFrame(
    pca.components_.T,
    index=ORDERED_PRODUCTS,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)]
)
loading_df['category'] = [PRODUCT_CATEGORY[p] for p in ORDERED_PRODUCTS]

print('Top 10 PCs — dominant category by mean |loading|:')
print(f'{"PC":<6} {"var%":>6}  {"dominant_cat":<20} {"mean|load|":>10}  {"2nd_cat":<20} {"mean|load|":>10}')
for pc_idx in range(10):
    pc_col = f'PC{pc_idx+1}'
    abs_loads = loading_df[pc_col].abs()
    abs_loads_df = pd.DataFrame({'abs_load': abs_loads, 'category': loading_df['category']})
    cat_means = abs_loads_df.groupby('category')['abs_load'].mean().sort_values(ascending=False)
    top1 = cat_means.index[0]
    top2 = cat_means.index[1] if len(cat_means) > 1 else '—'
    ev = explained[pc_idx]
    print(f'  PC{pc_idx+1:<3} {ev*100:>5.2f}%  {top1:<20} {cat_means[top1]:>10.5f}  {top2:<20} {cat_means[top2]:>10.5f}')


Top 10 PCs — dominant category by mean |loading|:
PC       var%  dominant_cat         mean|load|  2nd_cat              mean|load|
  PC1   12.70%  PEBBLES                 0.39083  UV_VISOR                0.00264
  PC2    5.80%  PEBBLES                 0.38059  MICROCHIP               0.00799
  PC3    4.28%  ROBOT                   0.20166  PEBBLES                 0.03000
  PC4    3.84%  PEBBLES                 0.37738  ROBOT                   0.02093
  PC5    3.37%  MICROCHIP               0.36064  PEBBLES                 0.01646
  PC6    3.31%  MICROCHIP               0.34971  PEBBLES                 0.08352
  PC7    3.30%  MICROCHIP               0.31118  PEBBLES                 0.02065
  PC8    3.27%  MICROCHIP               0.26745  PEBBLES                 0.09267
  PC9    3.20%  PEBBLES                 0.26166  MICROCHIP               0.14365
  PC10   2.11%  SNACKPACK               0.34228  UV_VISOR                0.02605


## 11. Per-Product Loading Magnitude — Top Drivers of Each PC

For each of PC1–PC3, list the top 10 products by absolute loading value.


In [15]:
for pc_num in [1, 2, 3]:
    pc_col = f'PC{pc_num}'
    top = loading_df[pc_col].abs().sort_values(ascending=False).head(10)
    print(f'\n--- {pc_col} top 10 |loadings| ---')
    for prod, val in top.items():
        actual_load = loading_df.loc[prod, pc_col]
        cat = PRODUCT_CATEGORY[prod]
        print(f'  {prod:<45}  loading={actual_load:+.5f}  |load|={val:.5f}  [{cat}]')



--- PC1 top 10 |loadings| ---
  PEBBLES_XL                                     loading=+0.78434  |load|=0.78434  [PEBBLES]
  PEBBLES_XS                                     loading=-0.46758  |load|=0.46758  [PEBBLES]
  PEBBLES_S                                      loading=-0.26649  |load|=0.26649  [PEBBLES]
  PEBBLES_M                                      loading=-0.22191  |load|=0.22191  [PEBBLES]
  PEBBLES_L                                      loading=-0.21381  |load|=0.21381  [PEBBLES]
  TRANSLATOR_VOID_BLUE                           loading=+0.00500  |load|=0.00500  [TRANSLATOR]
  UV_VISOR_RED                                   loading=+0.00445  |load|=0.00445  [UV_VISOR]
  GALAXY_SOUNDS_SOLAR_WINDS                      loading=+0.00389  |load|=0.00389  [GALAXY_SOUNDS]
  MICROCHIP_SQUARE                               loading=-0.00334  |load|=0.00334  [MICROCHIP]
  TRANSLATOR_ECLIPSE_CHARCOAL                    loading=-0.00333  |load|=0.00333  [TRANSLATOR]

--- PC2 top 10 |loading

## 12. PCA Biplot — PC1 vs PC2 (Products Colored by Category)

Each point = one product. Tight clusters = category cohesion. Outliers = products with idiosyncratic factor exposure.


In [16]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
pc_pairs = [(1, 2), (2, 3)]

for ax, (pc_a, pc_b) in zip(axes, pc_pairs):
    col_a, col_b = f'PC{pc_a}', f'PC{pc_b}'
    for prod in ORDERED_PRODUCTS:
        cat = PRODUCT_CATEGORY[prod]
        xa = loading_df.loc[prod, col_a]
        xb = loading_df.loc[prod, col_b]
        short_name = prod.split('_')[-1][:6]
        ax.scatter(xa, xb, color=CAT_COLORS[cat], s=60, zorder=3, alpha=0.85)
        ax.annotate(short_name, (xa, xb), fontsize=5.5, ha='center', va='bottom')
    
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.set_xlabel(f'{col_a} ({explained[pc_a-1]*100:.2f}%)')
    ax.set_ylabel(f'{col_b} ({explained[pc_b-1]*100:.2f}%)')
    ax.set_title(f'PCA Biplot: {col_a} vs {col_b}')

handles = [mpatches.Patch(color=CAT_COLORS[c], label=c) for c in CAT_COLORS]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=8, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig('/Users/bensinek/Documents/Coding/Prosperity4/notebooks/round_5/plots/06_pca_biplot.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close()


## 13. Day-by-Day Stability — Does the Correlation Structure Hold Across Days?

Compute the 50×50 return correlation matrix per day separately, then compare. If correlations are stable, the factor structure is reliable. If they differ, the structure may be day-specific noise.


In [17]:
per_day_corr = {}
per_day_explained = {}

for d in DAYS:
    ret_d = returns_frames[DAYS.index(d)][ORDERED_PRODUCTS]
    corr_d = ret_d.corr()
    per_day_corr[d] = corr_d
    
    pca_d = PCA(n_components=10)
    pca_d.fit(ret_d.fillna(0).values)
    per_day_explained[d] = pca_d.explained_variance_ratio_

print('Per-day PC1 explained variance:')
for d in DAYS:
    print(f'  Day {d}: PC1={per_day_explained[d][0]*100:.2f}%  PC2={per_day_explained[d][1]*100:.2f}%  PC3={per_day_explained[d][2]*100:.2f}%')

# Frobenius distance between per-day correlation matrices
from itertools import combinations
print('\nFrobenius norm of difference between per-day correlation matrices:')
for d1, d2 in combinations(DAYS, 2):
    diff = per_day_corr[d1] - per_day_corr[d2]
    frob = np.linalg.norm(diff.values, 'fro')
    print(f'  Day {d1} vs Day {d2}: ||C_{d1} - C_{d2}||_F = {frob:.4f}')

# Average |correlation| per day  
print('\nAverage |within-category return correlation| per day:')
for d in DAYS:
    corr_d = per_day_corr[d]
    within_vals = []
    for cat, prods in CATEGORIES.items():
        for i, pi in enumerate(prods):
            for j, pj in enumerate(prods):
                if i < j:
                    within_vals.append(abs(corr_d.loc[pi, pj]))
    print(f'  Day {d}: mean |within-cat corr| = {np.mean(within_vals):.4f}  '
          f'min={np.min(within_vals):.4f}  max={np.max(within_vals):.4f}')


Per-day PC1 explained variance:
  Day 2: PC1=14.50%  PC2=4.07%  PC3=3.62%
  Day 3: PC1=12.82%  PC2=6.14%  PC3=3.85%
  Day 4: PC1=11.98%  PC2=9.21%  PC3=6.49%

Frobenius norm of difference between per-day correlation matrices:
  Day 2 vs Day 3: ||C_2 - C_3||_F = 0.6895
  Day 2 vs Day 4: ||C_2 - C_4||_F = 0.7203
  Day 3 vs Day 4: ||C_3 - C_4||_F = 0.7169

Average |within-category return correlation| per day:
  Day 2: mean |within-cat corr| = 0.0636  min=0.0000  max=0.9316
  Day 3: mean |within-cat corr| = 0.0656  min=0.0002  max=0.9211
  Day 4: mean |within-cat corr| = 0.0655  min=0.0001  max=0.9173


## 14. High Cross-Category Pairs

Identify specific pairs of products from DIFFERENT categories with |return correlation| > 0.3 (arbitrary threshold for flagging). These are the most interesting cross-category linkages.


In [18]:
THRESHOLD = 0.3
cross_pairs = []
for i, pi in enumerate(ORDERED_PRODUCTS):
    for j, pj in enumerate(ORDERED_PRODUCTS):
        if i >= j:
            continue
        cat_i = PRODUCT_CATEGORY[pi]
        cat_j = PRODUCT_CATEGORY[pj]
        if cat_i == cat_j:
            continue
        r = corr_returns.loc[pi, pj]
        if abs(r) >= THRESHOLD:
            cross_pairs.append({'product_1': pi, 'category_1': cat_i,
                                 'product_2': pj, 'category_2': cat_j,
                                 'return_corr': r})

if cross_pairs:
    cross_df = pd.DataFrame(cross_pairs)
    cross_df['abs_corr'] = cross_df['return_corr'].abs()
    cross_df = cross_df.sort_values('abs_corr', ascending=False).drop(columns='abs_corr')
    print(f'Cross-category pairs with |return_corr| >= {THRESHOLD}: {len(cross_df)}')
    print(cross_df.to_string(index=False))
else:
    cross_df = pd.DataFrame(columns=['product_1','category_1','product_2','category_2','return_corr'])
    print(f'Cross-category pairs with |return_corr| >= {THRESHOLD}: 0')
    print('None found — no meaningful cross-category return correlations above threshold.')


Cross-category pairs with |return_corr| >= 0.3: 0
None found — no meaningful cross-category return correlations above threshold.


## 15. Within-Category Correlation — All Pairwise Values

Print all pairwise within-category return correlations, sorted by category and magnitude. This is the direct input for notebook 05's corroboration.


In [19]:
print(f'{"Category":<16} {"Product A":<42} {"Product B":<42} {"ret_corr":>9}  {"lvl_corr":>9}')
print('-' * 125)
for cat, prods in sorted(CATEGORIES.items()):
    pairs = [(prods[i], prods[j]) for i in range(len(prods)) for j in range(i+1, len(prods))]
    pair_data = [(pi, pj, corr_returns.loc[pi, pj], corr_levels.loc[pi, pj]) for pi, pj in pairs]
    pair_data.sort(key=lambda x: -x[2])
    for pi, pj, r_ret, r_lvl in pair_data:
        print(f'{cat:<16} {pi:<42} {pj:<42} {r_ret:>9.5f}  {r_lvl:>9.5f}')
    print()


Category         Product A                                  Product B                                   ret_corr   lvl_corr
-----------------------------------------------------------------------------------------------------------------------------
GALAXY_SOUNDS    GALAXY_SOUNDS_PLANETARY_RINGS              GALAXY_SOUNDS_SOLAR_FLAMES                   0.01298   -0.29690
GALAXY_SOUNDS    GALAXY_SOUNDS_BLACK_HOLES                  GALAXY_SOUNDS_SOLAR_WINDS                    0.00896   -0.32306
GALAXY_SOUNDS    GALAXY_SOUNDS_DARK_MATTER                  GALAXY_SOUNDS_PLANETARY_RINGS                0.00744    0.23648
GALAXY_SOUNDS    GALAXY_SOUNDS_DARK_MATTER                  GALAXY_SOUNDS_SOLAR_FLAMES                   0.00448   -0.15527
GALAXY_SOUNDS    GALAXY_SOUNDS_SOLAR_WINDS                  GALAXY_SOUNDS_SOLAR_FLAMES                   0.00403   -0.17812
GALAXY_SOUNDS    GALAXY_SOUNDS_BLACK_HOLES                  GALAXY_SOUNDS_SOLAR_FLAMES                   0.00194    0.11060
GALAXY

## 16. Summary Findings Table

Consolidated per-category summary for the findings doc.


In [20]:
print('=== FINAL SUMMARY TABLE ===\n')
print(f'{"Category":<16} {"within_ret_mean":>16} {"within_ret_min":>15} {"within_lvl_mean":>16} {"cross_max_abs_ret":>18} {"PCA_dominant_PC":>16}')
print('-' * 100)

# Find which PC is most dominated by each category
cat_dominant_pc = {}
for cat in CATEGORIES:
    best_pc = None
    best_mean = 0
    for pc_idx in range(10):
        pc_col = f'PC{pc_idx+1}'
        abs_loads = loading_df[pc_col].abs()
        abs_loads_df = pd.DataFrame({'abs_load': abs_loads, 'category': loading_df['category']})
        cat_means = abs_loads_df.groupby('category')['abs_load'].mean()
        if cat_means[cat] > best_mean:
            best_mean = cat_means[cat]
            best_pc = pc_idx + 1
    cat_dominant_pc[cat] = (best_pc, best_mean)

for cat in sorted(CATEGORIES.keys()):
    wm_r = summary.loc[cat, 'within_mean_ret']
    wmin_r = summary.loc[cat, 'within_min_ret']
    wm_l = summary.loc[cat, 'within_mean_lvl']
    cx_r = summary.loc[cat, 'cross_max_abs_ret']
    dpc, dpc_val = cat_dominant_pc[cat]
    print(f'{cat:<16} {wm_r:>16.5f} {wmin_r:>15.5f} {wm_l:>16.5f} {cx_r:>18.5f} {"PC"+str(dpc):>16}  (mean|load|={dpc_val:.5f})')


=== FINAL SUMMARY TABLE ===

Category          within_ret_mean  within_ret_min  within_lvl_mean  cross_max_abs_ret  PCA_dominant_PC
----------------------------------------------------------------------------------------------------
GALAXY_SOUNDS             0.00400        -0.00128         -0.05324            0.02539             PC10  (mean|load|=0.01544)
MICROCHIP                 0.00617         0.00180         -0.01873            0.01561              PC5  (mean|load|=0.36064)
OXYGEN_SHAKE              0.00702         0.00280         -0.07837            0.02796             PC10  (mean|load|=0.01457)
PANEL                     0.00450        -0.00845         -0.05229            0.02024             PC10  (mean|load|=0.00856)
PEBBLES                  -0.19128        -0.50593         -0.13921            0.01782              PC1  (mean|load|=0.39083)
ROBOT                    -0.00085        -0.00960         -0.04977            0.02429              PC3  (mean|load|=0.20166)
SLEEP_POD        